In [ ]:
import random
import math
import openpyxl



FILE_PATH = "/content/asymmetric_cipher_dataset.xlsx"
UID = 2024300251


random.seed(UID)




def is_prime(n):

    if n < 2:
        return False

    for i in range(2, int(math.sqrt(n)) + 1):

        if n % i == 0:
            return False

    return True


# ============================================================
# 3. GENERATE A RANDOM PRIME
# ============================================================

def generate_prime():

    while True:

        # Generate a random number
        candidate = random.randint(10000, 50000)

        # Check if it is prime
        if is_prime(candidate):
            return candidate


# ============================================================
# 4. GENERATE RSA PARAMETERS
# ============================================================

def generate_rsa_parameters():

    # Generate two different prime numbers
    p = generate_prime()
    q = generate_prime()

    while q == p:
        q = generate_prime()

    # Calculate n
    n = p * q

    # Calculate Euler's Totient
    phi_n = (p - 1) * (q - 1)

    # Choose e
    e = 65537

    # Make sure e is valid
    if e >= phi_n or math.gcd(e, phi_n) != 1:

        e = 3

        while math.gcd(e, phi_n) != 1:
            e = e + 2

    # Calculate d
    d = pow(e, -1, phi_n)

    return p, q, n, phi_n, e, d


# ============================================================
# 5. RSA ENCRYPTION
# ============================================================

def rsa_encrypt(message, e, n):

    # c = m^e mod n
    ciphertext = pow(message, e, n)

    return ciphertext


# ============================================================
# 6. RSA DECRYPTION
# ============================================================

def rsa_decrypt(ciphertext, d, n):

    # m = c^d mod n
    decrypted_message = pow(ciphertext, d, n)

    return decrypted_message


# ============================================================
# 7. OPEN EXCEL FILE
# ============================================================

workbook = openpyxl.load_workbook(FILE_PATH)

rsa_sheet = workbook["RSA"]


# ============================================================
# 8. STORE UID
# ============================================================

instructions_sheet = workbook["Instructions"]

instructions_sheet["B4"] = UID


# ============================================================
# 9. PROCESS ALL RSA MESSAGES
# ============================================================

for row in range(2, rsa_sheet.max_row + 1):

    # Read message from column B
    message = rsa_sheet.cell(row=row, column=2).value

    if message is None:
        continue

    try:
        message = int(message)
    except ValueError:
        continue
    # --------------------------------------------------------
    # Generate RSA parameters
    # --------------------------------------------------------

    p, q, n, phi_n, e, d = generate_rsa_parameters()

    # --------------------------------------------------------
    # Check that message is smaller than n
    # --------------------------------------------------------

    if message >= n:
        print("Message is too large for RSA at row:", row)
        continue

    # --------------------------------------------------------
    # Encrypt
    # --------------------------------------------------------

    ciphertext = rsa_encrypt(message, e, n)

    # --------------------------------------------------------
    # Decrypt
    # --------------------------------------------------------

    decrypted_message = rsa_decrypt(ciphertext, d, n)

    # --------------------------------------------------------
    # Check whether decryption was successful
    # --------------------------------------------------------

    decryption_ok = (message == decrypted_message)

    # --------------------------------------------------------
    # Write results into Excel
    # --------------------------------------------------------

    # Column C → p
    rsa_sheet.cell(row=row, column=3).value = p

    # Column D → q
    rsa_sheet.cell(row=row, column=4).value = q

    # Column E → n
    rsa_sheet.cell(row=row, column=5).value = n

    # Column F → phi(n)
    rsa_sheet.cell(row=row, column=6).value = phi_n

    # Column G → e
    rsa_sheet.cell(row=row, column=7).value = e

    # Column H → d
    rsa_sheet.cell(row=row, column=8).value = d

    # Column I → Ciphertext
    rsa_sheet.cell(row=row, column=9).value = ciphertext

    # Column J → Decrypted Message
    rsa_sheet.cell(row=row, column=10).value = decrypted_message

    # Column K → Decryption OK
    rsa_sheet.cell(row=row, column=11).value = decryption_ok


# ============================================================
# 10. SAVE EXCEL FILE
# ============================================================

workbook.save(FILE_PATH)

print("RSA processing completed successfully!")
print("UID:", UID)
print("File updated:", FILE_PATH)

RSA processing completed successfully!
UID: 2024300251
File updated: /content/asymmetric_cipher_dataset.xlsx


In [ ]:
import random
import math
import openpyxl


# ============================================================
# 1. SETTINGS
# ============================================================

FILE_PATH = "/content/asymmetric_cipher_dataset.xlsx"
UID = 2024300251

# Use UID as the random seed
random.seed(UID)


# ============================================================
# 2. CHECK IF A NUMBER IS PRIME
# ============================================================

def is_prime(n):

    if n < 2:
        return False

    for i in range(2, int(math.sqrt(n)) + 1):

        if n % i == 0:
            return False

    return True


# ============================================================
# 3. GENERATE A RANDOM PRIME
# ============================================================

def generate_prime():

    while True:

        candidate = random.randint(10000, 50000)

        if is_prime(candidate):
            return candidate


# ============================================================
# 4. FIND A GENERATOR
# ============================================================

def find_generator(p):

    # For this lab implementation, try small values
    # and find a generator of the multiplicative group.

    for g in range(2, p):

        # A simple check suitable for our lab-sized prime.
        # We want g to have a large order.

        if pow(g, (p - 1) // 2, p) != 1:
            return g

    return 2


# ============================================================
# 5. GENERATE ELGAMAL PARAMETERS
# ============================================================

def generate_elgamal_parameters():

    # Generate prime p
    p = generate_prime()

    # Generate generator g
    g = find_generator(p)

    # Generate private key x
    x = random.randint(2, p - 2)

    # Calculate public component
    # y = g^x mod p
    y = pow(g, x, p)

    return p, g, x, y


# ============================================================
# 6. ELGAMAL ENCRYPTION
# ============================================================

def elgamal_encrypt(message, p, g, y):

    # Generate a new random k for every message
    k = random.randint(2, p - 2)

    # First ciphertext component
    # c1 = g^k mod p
    c1 = pow(g, k, p)

    # Second ciphertext component
    # c2 = message * y^k mod p
    c2 = (message * pow(y, k, p)) % p

    return k, c1, c2


# ============================================================
# 7. ELGAMAL DECRYPTION
# ============================================================

def elgamal_decrypt(c1, c2, x, p):

    # Calculate shared value
    # s = c1^x mod p
    s = pow(c1, x, p)

    # Find modular inverse of s
    s_inverse = pow(s, -1, p)

    # Recover original message
    # message = c2 * s^(-1) mod p
    message = (c2 * s_inverse) % p

    return message


# ============================================================
# 8. OPEN EXCEL FILE
# ============================================================

workbook = openpyxl.load_workbook(FILE_PATH)

elgamal_sheet = workbook["ElGamal"]


# ============================================================
# 9. STORE UID
# ============================================================

instructions_sheet = workbook["Instructions"]

instructions_sheet["B4"] = UID


# ============================================================
# 10. GENERATE ELGAMAL PARAMETERS
# ============================================================

p, g, x, y = generate_elgamal_parameters()

print("ElGamal Parameters")
print("-------------------")
print("p =", p)
print("g =", g)
print("Private x =", x)
print("Public y =", y)


# ============================================================
# 11. PROCESS ALL ELGAMAL MESSAGES
# ============================================================

for row in range(2, elgamal_sheet.max_row + 1):

    # Read message from column B
    message = elgamal_sheet.cell(row=row, column=2).value

    # Skip empty rows
    if message is None:
        continue

    # Skip instruction/text rows
    try:
        message = int(message)
    except (ValueError, TypeError):
        continue

    # ========================================================
    # Message must be smaller than p
    # ========================================================

    if message >= p:

        print(
            f"Skipping row {row}: "
            f"message {message} is greater than or equal to p {p}"
        )

        continue

    # ========================================================
    # Encrypt
    # ========================================================

    k, c1, c2 = elgamal_encrypt(
        message,
        p,
        g,
        y
    )

    # ========================================================
    # Decrypt
    # ========================================================

    decrypted_message = elgamal_decrypt(
        c1,
        c2,
        x,
        p
    )

    # ========================================================
    # Verify
    # ========================================================

    decryption_ok = (message == decrypted_message)

    # ========================================================
    # WRITE RESULTS INTO EXCEL
    # ========================================================

    # Column C → p
    elgamal_sheet.cell(row=row, column=3).value = p

    # Column D → g
    elgamal_sheet.cell(row=row, column=4).value = g

    # Column E → Private x
    elgamal_sheet.cell(row=row, column=5).value = x

    # Column F → Public y
    elgamal_sheet.cell(row=row, column=6).value = y

    # Column G → k
    elgamal_sheet.cell(row=row, column=7).value = k

    # Column H → Ciphertext c1
    elgamal_sheet.cell(row=row, column=8).value = c1

    # Column I → Ciphertext c2
    elgamal_sheet.cell(row=row, column=9).value = c2

    # Column J → Decrypted Message
    elgamal_sheet.cell(row=row, column=10).value = decrypted_message

    # Column K → Decryption OK
    elgamal_sheet.cell(row=row, column=11).value = decryption_ok


# ============================================================
# 12. SAVE UPDATED WORKBOOK
# ============================================================

workbook.save(FILE_PATH)


# ============================================================
# 13. VERIFY RESULTS
# ============================================================

print()
print("ElGamal processing completed!")
print("UID:", UID)
print("File:", FILE_PATH)


all_ok = True

for row in range(2, elgamal_sheet.max_row + 1):

    message = elgamal_sheet.cell(row=row, column=2).value

    try:
        message = int(message)
    except (ValueError, TypeError):
        continue

    result = elgamal_sheet.cell(row=row, column=11).value

    if result != True:

        all_ok = False

        print(
            f"ElGamal failed at row {row}"
        )


print("All ElGamal decryptions correct:", all_ok)

ElGamal Parameters
-------------------
p = 36293
g = 2
Private x = 15230
Public y = 24462

ElGamal processing completed successfully!
UID: 2024300251
File: /content/asymmetric_cipher_dataset_RSA_DH_completed.xlsx
All ElGamal decryptions correct: True


In [ ]:
import random
import math
import openpyxl


# ============================================================
# 1. SETTINGS
# ============================================================

FILE_PATH = "/content/asymmetric_cipher_dataset.xlsx"
UID = 2024300251

# Use UID as the random seed
random.seed(UID)


# ============================================================
# 2. CHECK IF A NUMBER IS PRIME
# ============================================================

def is_prime(n):

    if n < 2:
        return False

    for i in range(2, int(math.sqrt(n)) + 1):

        if n % i == 0:
            return False

    return True


# ============================================================
# 3. GENERATE A RANDOM PRIME
# ============================================================

def generate_prime():

    while True:

        candidate = random.randint(10000, 50000)

        if is_prime(candidate):
            return candidate


# ============================================================
# 4. FIND A GENERATOR
# ============================================================

def find_generator(p):

    for g in range(2, p):

        if pow(g, (p - 1) // 2, p) != 1:
            return g

    return 2


# ============================================================
# 5. GENERATE DIFFIE-HELLMAN PARAMETERS
# ============================================================

def generate_dh_parameters():

    # Generate common public prime
    p = generate_prime()

    # Generate common public generator
    g = find_generator(p)

    # Generate Alice's private value
    a = random.randint(2, p - 2)

    # Generate Bob's private value
    b = random.randint(2, p - 2)

    return p, g, a, b


# ============================================================
# 6. CALCULATE PUBLIC VALUES
# ============================================================

def calculate_public_value(private_value, g, p):

    # Public value = g^private_value mod p
    public_value = pow(g, private_value, p)

    return public_value


# ============================================================
# 7. CALCULATE SHARED SECRET
# ============================================================

def calculate_shared_secret(other_public_value, private_value, p):

    # Shared secret = other_public_value^private_value mod p
    shared_secret = pow(
        other_public_value,
        private_value,
        p
    )

    return shared_secret


# ============================================================
# 8. OPEN EXCEL FILE
# ============================================================

workbook = openpyxl.load_workbook(FILE_PATH)

dh_sheet = workbook["Diffie-Hellman"]


# ============================================================
# 9. STORE UID
# ============================================================

instructions_sheet = workbook["Instructions"]

instructions_sheet["B4"] = UID


# ============================================================
# 10. GENERATE DH PARAMETERS
# ============================================================

p, g, a, b = generate_dh_parameters()


# ============================================================
# 11. CALCULATE ALICE'S PUBLIC VALUE
# ============================================================

A = calculate_public_value(
    a,
    g,
    p
)


# ============================================================
# 12. CALCULATE BOB'S PUBLIC VALUE
# ============================================================

B = calculate_public_value(
    b,
    g,
    p
)


# ============================================================
# 13. CALCULATE ALICE'S SHARED SECRET
# ============================================================

alice_secret = calculate_shared_secret(
    B,
    a,
    p
)


# ============================================================
# 14. CALCULATE BOB'S SHARED SECRET
# ============================================================

bob_secret = calculate_shared_secret(
    A,
    b,
    p
)


# ============================================================
# 15. CHECK WHETHER SECRETS MATCH
# ============================================================

secret_match = (alice_secret == bob_secret)


# ============================================================
# 16. PRINT RESULTS
# ============================================================

print("Diffie-Hellman Parameters")
print("--------------------------")

print("p =", p)
print("g =", g)

print()

print("Alice")
print("Private =", a)
print("Public  =", A)

print()

print("Bob")
print("Private =", b)
print("Public  =", B)

print()

print("Alice Shared Secret =", alice_secret)
print("Bob Shared Secret   =", bob_secret)

print()

print("Secrets Match:", secret_match)


# ============================================================
# 17. WRITE RESULTS INTO EXCEL
# ============================================================

# We need to find the row containing the first actual data entry.
# The workbook may contain instruction text in some cells.

data_row = None

for row in range(2, dh_sheet.max_row + 1):

    value = dh_sheet.cell(row=row, column=1).value

    if value is not None:

        try:
            int(value)
            data_row = row
            break

        except (ValueError, TypeError):
            continue


# ============================================================
# 18. WRITE DH RESULTS
# ============================================================

if data_row is not None:

    # Column B → p
    dh_sheet.cell(
        row=data_row,
        column=2
    ).value = p

    # Column C → g
    dh_sheet.cell(
        row=data_row,
        column=3
    ).value = g

    # Column D → Alice Private
    dh_sheet.cell(
        row=data_row,
        column=4
    ).value = a

    # Column E → Alice Public
    dh_sheet.cell(
        row=data_row,
        column=5
    ).value = A

    # Column F → Bob Private
    dh_sheet.cell(
        row=data_row,
        column=6
    ).value = b

    # Column G → Bob Public
    dh_sheet.cell(
        row=data_row,
        column=7
    ).value = B

    # Column H → Alice Secret
    dh_sheet.cell(
        row=data_row,
        column=8
    ).value = alice_secret

    # Column I → Bob Secret
    dh_sheet.cell(
        row=data_row,
        column=9
    ).value = bob_secret

    # Column J → Match?
    dh_sheet.cell(
        row=data_row,
        column=10
    ).value = secret_match


# ============================================================
# 19. SAVE WORKBOOK
# ============================================================

workbook.save(FILE_PATH)


print()
print("Diffie-Hellman processing completed!")
print("UID:", UID)
print("File updated:", FILE_PATH)

Diffie-Hellman Parameters
--------------------------
p = 36293
g = 2

Alice
Private = 15230
Public  = 24462

Bob
Private = 32105
Public  = 17404

Alice Shared Secret = 31372
Bob Shared Secret   = 31372

Secrets Match: True

Diffie-Hellman processing completed!
UID: 2024300251
File updated: /content/asymmetric_cipher_dataset.xlsx


In [ ]:
import random
import openpyxl


# ============================================================
# 1. SETTINGS
# ============================================================

FILE_PATH = "/content/asymmetric_cipher_dataset.xlsx"
UID = 2024300251

random.seed(UID)


# ============================================================
# 2. ECC CURVE
# ============================================================

# Curve:
#
# y^2 = x^3 + 2x + 2 (mod 17)

p = 17
a = 2
b = 2

# Base point
G = (5, 1)

# Order of G for this curve
order = 19


# ============================================================
# 3. POINT ADDITION
# ============================================================

def point_add(P, Q):

    # If P is the point at infinity
    if P is None:
        return Q

    # If Q is the point at infinity
    if Q is None:
        return P

    x1, y1 = P
    x2, y2 = Q

    # P + (-P) = point at infinity
    if x1 == x2 and (y1 + y2) % p == 0:
        return None

    # --------------------------------------------------------
    # Point doubling
    # --------------------------------------------------------

    if P == Q:

        if y1 == 0:
            return None

        numerator = 3 * x1 * x1 + a
        denominator = 2 * y1

        denominator_inverse = pow(
            denominator,
            -1,
            p
        )

        slope = (
            numerator * denominator_inverse
        ) % p

    # --------------------------------------------------------
    # Normal point addition
    # --------------------------------------------------------

    else:

        numerator = y2 - y1
        denominator = x2 - x1

        denominator_inverse = pow(
            denominator,
            -1,
            p
        )

        slope = (
            numerator * denominator_inverse
        ) % p

    # Calculate x3
    x3 = (
        slope * slope
        - x1
        - x2
    ) % p

    # Calculate y3
    y3 = (
        slope * (x1 - x3)
        - y1
    ) % p

    return (x3, y3)


# ============================================================
# 4. SCALAR MULTIPLICATION
# ============================================================

def scalar_multiply(k, point):

    result = None

    for i in range(k):

        result = point_add(
            result,
            point
        )

    return result


# ============================================================
# 5. OPEN EXCEL FILE
# ============================================================

workbook = openpyxl.load_workbook(FILE_PATH)

ecc_sheet = workbook["ECC"]


# ============================================================
# 6. STORE UID
# ============================================================

instructions_sheet = workbook["Instructions"]

instructions_sheet["B4"] = UID


# ============================================================
# 7. WRITE CURVE INFORMATION IN THE FIRST DATA/TEMPLATE ROW
# ============================================================

# Row 2 is currently the curve/template row in your workbook.

ecc_sheet.cell(row=2, column=2).value = (
    "y^2 = x^3 + 2x + 2 mod 17"
)

ecc_sheet.cell(row=2, column=3).value = str(G)


# ============================================================
# 8. PROCESS ALL ACTUAL MESSAGE ROWS
# ============================================================

for row in range(3, ecc_sheet.max_row + 1):

    # Read message from column B
    message = ecc_sheet.cell(
        row=row,
        column=2
    ).value

    # Skip empty rows
    if message is None:
        continue

    # Skip text/instruction rows
    try:
        message = int(message)

    except (ValueError, TypeError):
        continue


    # ========================================================
    # Generate UID-based private key
    # ========================================================

    # d must be between 1 and order - 1
    d = random.randint(
        1,
        order - 1
    )


    # ========================================================
    # Calculate public key
    # ========================================================

    # Q = dG

    Q = scalar_multiply(
        d,
        G
    )


    # ========================================================
    # Perform ECC point operation again for verification
    # ========================================================

    point_operation_result = scalar_multiply(
        d,
        G
    )


    # ========================================================
    # Verify public key
    # ========================================================

    verification_result = (
        Q == point_operation_result
    )


    # ========================================================
    # Convert verification to Y/N
    # ========================================================

    if verification_result:
        correct = "Y"
    else:
        correct = "N"


    # ========================================================
    # WRITE RESULTS INTO EXCEL
    # ========================================================

    # Column C → Curve
    ecc_sheet.cell(
        row=row,
        column=3
    ).value = "y^2 = x^3 + 2x + 2 mod 17"


    # Column D → Base Point G
    ecc_sheet.cell(
        row=row,
        column=4
    ).value = str(G)


    # Column E → Private Key d
    ecc_sheet.cell(
        row=row,
        column=5
    ).value = d


    # Column F → Public Key Q
    ecc_sheet.cell(
        row=row,
        column=6
    ).value = str(Q)


    # Column G → Point Operation Result
    ecc_sheet.cell(
        row=row,
        column=7
    ).value = str(point_operation_result)


    # Column H → Verification Result
    ecc_sheet.cell(
        row=row,
        column=8
    ).value = verification_result


    # Column I → Correct Y/N
    ecc_sheet.cell(
        row=row,
        column=9
    ).value = correct


# ============================================================
# 9. SAVE FILE
# ============================================================

workbook.save(FILE_PATH)


# ============================================================
# 10. VERIFY RESULTS
# ============================================================

print("ECC processing completed!")
print("UID:", UID)
print("File updated:", FILE_PATH)

all_correct = True

for row in range(3, ecc_sheet.max_row + 1):

    message = ecc_sheet.cell(
        row=row,
        column=2
    ).value

    try:
        int(message)

    except (ValueError, TypeError):
        continue

    result = ecc_sheet.cell(
        row=row,
        column=9
    ).value

    if result != "Y":

        all_correct = False

        print(
            "ECC verification failed at row:",
            row
        )


print()
print("All ECC results correct:", all_correct)

ECC processing completed!
UID: 2024300251
File updated: /content/asymmetric_cipher_dataset.xlsx

All ECC results correct: True


In [ ]:
import pandas as pd

# Load the Excel file
excel_file = pd.ExcelFile(FILE_PATH)

# Read each sheet into a DataFrame and display the first few rows
for sheet_name in excel_file.sheet_names:
    print(f"\n--- Sheet: {sheet_name} ---")
    df_sheet = pd.read_excel(excel_file, sheet_name=sheet_name)
    display(df_sheet.head())


--- Sheet: Instructions ---


,Unnamed: 0,Unnamed: 1
0,Comparative Analysis of Asymmetric Cryptograph...,NaN
1,NaN,NaN
2,Student UID,2.024300e+09
3,Student Name,NaN
4,NaN,NaN



--- Sheet: RSA ---


,Sr_No,Message,p,q,n,phi(n),e,d,Ciphertext,Decrypted_Message,Decryption_OK
0,1,7122,36293.0,16249.0,5.897250e+08,5.896724e+08,65537.0,106045121.0,105060758.0,7122.0,1.0
1,2,8355,15199.0,31643.0,4.809420e+08,4.808951e+08,65537.0,169891277.0,240713611.0,8355.0,1.0
2,3,1691,15287.0,44021.0,6.729490e+08,6.728897e+08,65537.0,157777993.0,47299635.0,1691.0,1.0
3,4,3114,23131.0,11003.0,2.545104e+08,2.544763e+08,65537.0,176526233.0,156466191.0,3114.0,1.0
4,5,3313,47653.0,32647.0,1.555727e+09,1.555647e+09,65537.0,474477497.0,258447480.0,3313.0,1.0



--- Sheet: ElGamal ---


,Sr_No,Message,p,g,Private_Key_x,Public_Key_y,Ephemeral_k,Ciphertext_C1,Ciphertext_C2,Decrypted_Message,Decryption_OK
0,1,7122,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,8355,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,1691,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,3114,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,3313,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


... (2 outputs left out)


--- Sheet: ECC ---


,Sr_No,Message,Curve,Base_Point_G,Private_Key_d,Public_Key_Q,Point_Operation_Result,Verification_Result,Correct (Y/N)
0,1,y^2 = x^3 + 2x + 2 mod 17,"(5, 1)",NaN,NaN,NaN,NaN,NaN,NaN
1,2,8355,y^2 = x^3 + 2x + 2 mod 17,"(5, 1)",6.0,"(16, 13)","(16, 13)",1.0,Y
2,3,1691,y^2 = x^3 + 2x + 2 mod 17,"(5, 1)",16.0,"(10, 11)","(10, 11)",1.0,Y
3,4,3114,y^2 = x^3 + 2x + 2 mod 17,"(5, 1)",15.0,"(3, 16)","(3, 16)",1.0,Y
4,5,3313,y^2 = x^3 + 2x + 2 mod 17,"(5, 1)",11.0,"(13, 10)","(13, 10)",1.0,Y



--- Sheet: Comparative Analysis ---


,Student UID,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,NaN,NaN,NaN,NaN,NaN
1,A. Mathematical Foundation,NaN,NaN,NaN,NaN
2,Property,RSA,ElGamal,Diffie-Hellman,ECC
3,Mathematical Basis,NaN,NaN,NaN,NaN
4,Uses Large Primes,NaN,NaN,NaN,NaN
